## LULC Temporal Stability

We can't do this yet because we only have 2 time points of LULC. I could run a single tile for all years as a test. This is a plan to assess temporal stability across LULC outputs.

For each tile:
1. Load all LULC assets for the 2000-2025 time range
2. Flip rate: count how many times each pixel changes. Flag pixels over a threshhold.
3. Reversals/Flicker: Count A→B→A patterns within 1–3 years, such as Tree→Grassland→Tree. These are almost always classification noise rather than real change, which makes this the most diagnostic single metric. Flag flickering pixels and note which classes over which years.
4. Implausible transitions. Build an annual transition matrix and flag transitions that are unlikely within one year, such as Built-up→Tree cover, Water→Built-up outside known reclamation, or Mangrove↔Cropland. Report their share of all change.
5. Temporal stability. For each pixel, record the percentage of years matching its modal class (modal share) and the number of distinct classes it takes over 26 years. A single real change also lowers modal share, so only flag low modal share on pixels with more than one change that are not flicker. This leaves long reversals (B lasting more than the flicker window) and multi-step change (A -> B -> C). Report mean modal share per modal class to compare class stability, and map distinct classes to find unstable hotspots, which typically fall in coastal mixed pixels, cloud-prone areas and grass/crop boundaries.
6. Class area time series. Plot annual area per class. Year-to-year jumps that coincide with input changes are artefacts, not land change. Common causes are Landsat 5→7→8/9 sensor transitions, Landsat 7 SLC-off gaps after 2003, and years with low clear-observation counts in the GeoMAD.
7. Invariant areas. Areas known to be stable, such as protected forest or old urban cores, should show close to zero change.

TODO: What do do if we see bad temporal stability for many pixels, especially for certain classes.

In [ ]:
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
# TODO: Run this for all test tiles? Not just 1 other tile.

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dep_tools.grids import PACIFIC_EPSG
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from odc.geo.geom import Geometry
from odc.geo.xr import rasterize
from odc.stac import load as stac_load
from PIL import Image
from pystac import Item
from rustac import search_sync
from scipy.stats import spearmanr

from ldn.aws import configure_s3_access_profile
from ldn.grids import EPSG_CODE, get_gadm
from ldn.raster import GEOMAD_BANDS, MAD_BANDS, REFLECTANCE_BANDS, _calculate_indices, _scale_offset_landsat
from ldn.typology import classes, classes_flipped, colors
from ldn.utils import (
    LULC_DATASET_ID,
    SENSOR,
    dataset_prefix,
)

configure_s3_access_profile()

In [ ]:
BUCKET_URL = "https://dep-public-staging.s3.us-west-2.amazonaws.com"
LULC_VERSION = "0-1-1"
OWNER = "dep"
LULC_PREFIX = dataset_prefix(OWNER, SENSOR, LULC_DATASET_ID)
LULC_PARQUET = f"{BUCKET_URL}/{LULC_PREFIX}/{LULC_VERSION}/{LULC_PREFIX}.parquet"
REPORT_DIR = Path("reports_temporal") / LULC_VERSION  # Relative to this notebook (notebooks/LULC_Testing)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TILES = [(50, 41)]  # Nauru
YEARS = range(2000, 2026)
NODATA = 255
COUNTRY = {"Nauru": "NRU"}  # Pixels outside this country are set to nodata
COUNTRY_BUFFER_M = 200

FLIP_THRESHOLD = 4  # flag pixels with more than this many year-to-year changes
FLICKER_MAX_YEARS = 3  # A -> B -> A where B lasts at most this many years
MODAL_THRESHOLD = 0.8  # flag long reversals or drift when the modal class covers less than this share of years

# Transitions that are unlikely within one year
IMPLAUSIBLE = [
    (classes["Built-up"], classes[label]) for label in ["Tree cover", "Grassland", "Cropland", "Wetland", "Water"]
] + [
    (classes["Water"], classes["Built-up"]),
    (classes["Wetland"], classes["Cropland"]),
    (classes["Cropland"], classes["Wetland"]),
]

In [ ]:
def load_tile(x: int, y: int):
    """Load the annual LULC classification for one tile as a (time, y, x) DataArray over the buffered COUNTRY,
    nodata outside it."""
    ids = [f"{LULC_PREFIX}_{x:03}_{y:03}_{year}" for year in YEARS]
    docs = search_sync(LULC_PARQUET, ids=ids)
    items = [Item.from_dict(doc) for doc in docs]
    if len(items) != len(ids):
        raise ValueError(f"Found {len(items)} of {len(ids)} items for tile {x:03}_{y:03}")
    # Load and mask to the buffered country to drop the open ocean that dominates Pacific tiles
    country = Geometry(get_gadm(countries=COUNTRY).geometry.union_all(), crs="EPSG:4326")
    buffered = country.to_crs(PACIFIC_EPSG).buffer(COUNTRY_BUFFER_M)
    lulc = stac_load(items, geopolygon=buffered, measurements=["classification"], chunks={})["classification"].load()
    return lulc.where(rasterize(buffered, lulc.odc.geobox), NODATA)


CLASS_CODES = [c for c in classes_flipped if c != NODATA]


def class_rgb(code: int) -> tuple[int, int, int]:
    """Typology colour of a class code as 0-255 RGB, from its "rgb(r,g,b)" string."""
    return tuple(int(v) for v in colors[code][4:-1].split(","))


def class_rgba() -> np.ndarray:
    """Lookup from class code to RGBA, as a (256, 4) uint8 array with nodata and unused codes transparent."""
    rgba = np.zeros((256, 4), np.uint8)
    for code in CLASS_CODES:
        rgba[code] = (*class_rgb(code), 255)
    return rgba


def forward_fill(a: np.ndarray) -> np.ndarray:
    """Fill nodata years with the previous valid year. Leading nodata stays nodata."""
    years = np.arange(len(a))[:, None, None]
    return np.take_along_axis(a, np.maximum.accumulate(np.where(a != NODATA, years, 0), axis=0), axis=0)


def fill_missing(a: np.ndarray) -> np.ndarray:
    """Fill nodata years with the previous valid year, or the next one for leading gaps.

    Year-to-year tests then compare each valid year with the previous valid year, skipping gaps. Pixels with no
    valid years stay nodata.

    Args:
        a: Class array of shape (time, y, x).

    Returns:
        Filled class array of the same shape.
    """
    return forward_fill(forward_fill(a)[::-1])[::-1]


def class_counts(a: np.ndarray) -> np.ndarray:
    """Count the years each pixel is in each class of CLASS_CODES, as an array of shape (class, y, x)."""
    return np.stack([(a == c).sum(axis=0) for c in CLASS_CODES])


def flips(a: np.ndarray) -> np.ndarray:
    """Count the year-to-year changes of each pixel."""
    return (a[1:] != a[:-1]).sum(axis=0)


def flip_rate(a: np.ndarray) -> np.ndarray:
    """Flag pixels with more than FLIP_THRESHOLD year-to-year changes."""
    return flips(a) > FLIP_THRESHOLD


def flicker_events(a: np.ndarray) -> np.ndarray:
    """Find A -> B -> A patterns where B lasts at most FLICKER_MAX_YEARS years.

    Each event is marked once, at the year A changes to B, using the first year the pixel returns to A.

    Args:
        a: Class array of shape (time, y, x).

    Returns:
        Boolean array of shape (time - 1, y, x).
    """
    unreturned = a[1:] != a[:-1]
    events = np.zeros_like(unreturned)
    for lag in range(2, FLICKER_MAX_YEARS + 2):
        returned = unreturned[: len(a) - lag] & (a[:-lag] == a[lag:])
        unreturned[: len(a) - lag] &= ~returned
        events[: len(a) - lag] |= returned
    return events


def flicker(events: np.ndarray) -> np.ndarray:
    """Flag pixels with any flicker event from flicker_events."""
    return events.any(axis=0)


def implausible_events(a: np.ndarray) -> np.ndarray:
    """Find year-to-year transitions in IMPLAUSIBLE.

    Args:
        a: Class array of shape (time, y, x).

    Returns:
        Boolean array of shape (time - 1, y, x), True at the year each transition starts.
    """
    lookup = np.zeros((256, 256), bool)
    for src, dst in IMPLAUSIBLE:
        lookup[src, dst] = True
    return lookup[a[:-1], a[1:]]


def implausible(events: np.ndarray) -> np.ndarray:
    """Flag pixels with any implausible transition from implausible_events."""
    return events.any(axis=0)


def modal_share(counts: np.ndarray) -> np.ndarray:
    """Share of years each pixel is in its modal class, from class_counts. NaN for pixels with no valid years."""
    total = counts.sum(axis=0)
    return np.divide(counts.max(axis=0), total, out=np.full(total.shape, np.nan), where=total > 0)


def distinct_classes(counts: np.ndarray) -> np.ndarray:
    """Count the distinct classes each pixel takes, from class_counts."""
    return (counts > 0).sum(axis=0)


def long_reversal_or_drift(a: np.ndarray, counts: np.ndarray, flickered: np.ndarray) -> np.ndarray:
    """Flag pixels with low modal share that is not a single change or flicker.

    A single real change lowers modal share, so only pixels with more than one change are flagged. Flicker is
    excluded as it has its own test. What remains are reversals longer than FLICKER_MAX_YEARS and multi-step
    change (A -> B -> C).

    Args:
        a: Class array of shape (time, y, x).
        counts: Class counts from class_counts.
        flickered: Pixels flagged by flicker.

    Returns:
        Boolean (y, x) array.
    """
    return (modal_share(counts) < MODAL_THRESHOLD) & (flips(a) > 1) & ~flickered


def class_stability(counts: np.ndarray, valid: np.ndarray) -> pd.DataFrame:
    """Summarise modal share by each pixel's modal class.

    Args:
        counts: Class counts from class_counts.
        valid: Boolean (y, x) array of valid pixels.

    Returns:
        DataFrame indexed by class name with the pixel count and mean modal share of valid pixels.
    """
    modal = counts.argmax(axis=0)[valid]
    pixels = np.bincount(modal, minlength=len(CLASS_CODES))
    share = np.bincount(modal, weights=modal_share(counts)[valid], minlength=len(CLASS_CODES))
    found = pixels > 0
    return pd.DataFrame(
        {"pixels": pixels[found], "mean_modal_share": (share[found] / pixels[found]).round(3)},
        index=[classes_flipped[c] for c in np.array(CLASS_CODES)[found]],
    )


def flag_pixels(a: np.ndarray, counts: np.ndarray, flickers: np.ndarray, implausibles: np.ndarray) -> dict:
    """Flag temporally unstable pixels for each test.

    Args:
        a: Class array of shape (time, y, x).
        counts: Class counts from class_counts.
        flickers: Flicker events from flicker_events.
        implausibles: Implausible transitions from implausible_events.

    Returns:
        Dict of boolean (y, x) arrays: "valid" plus one per test. Pixels with nodata in any year are
        not valid and are never flagged, so pass an array from fill_missing to only drop pixels with no data.
    """
    valid = (a != NODATA).all(axis=0)
    flickered = flicker(flickers)
    flags = {
        "flip_rate": flip_rate(a),
        "flicker": flickered,
        "implausible": implausible(implausibles),
        "long_reversal_or_drift": long_reversal_or_drift(a, counts, flickered),
    }
    flags = {name: flagged & valid for name, flagged in flags.items()}
    flags["any"] = np.logical_or.reduce(list(flags.values()))
    flags["valid"] = valid
    return flags


def categorise(codes: np.ndarray, valid: np.ndarray, known: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Categorise each pixel by the codes it has.

    Args:
        codes: Integer array of shape (n, y, x), -1 where there is no code.
        valid: Boolean (y, x) array of valid pixels.
        known: All possible codes, to list codes with no pixels. If None, only codes found are listed.

    Returns:
        (y, x) category array and the codes it refers to. Category 0 is no code, 1 to n is the matching
        returned code, n + 1 is more than one code and n + 2 is not valid.
    """
    first = np.where(codes >= 0, codes, np.iinfo(codes.dtype).max).min(axis=0)
    last = codes.max(axis=0)
    single = first == last
    unique = np.unique(last[single & valid] if known is None else known)
    category = np.select(
        [~valid, last < 0, ~single],
        [len(unique) + 2, 0, len(unique) + 1],
        np.searchsorted(unique, last) + 1,
    )
    return category, unique


def transition_categories(
    a: np.ndarray, events: np.ndarray, valid: np.ndarray, known: list[tuple[int, int]] | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """Categorise each pixel by the year-to-year transitions marked in events.

    Args:
        a: Class array of shape (time, y, x).
        events: Boolean array of shape (time - 1, y, x) marking the transitions to use.
        valid: Boolean (y, x) array of valid pixels.
        known: All possible (src, dst) transitions. If None, only transitions found are listed.

    Returns:
        Category array and codes as from categorise. Transitions are encoded as src * 256 + dst.
    """
    codes = np.where(events, a[:-1].astype(np.int32) * 256 + a[1:], -1)
    return categorise(codes, valid, None if known is None else np.array([src * 256 + dst for src, dst in known]))


def plot_categories(category: np.ndarray, labels: list[str], title: str, png: Path):
    """Map categories with the pixel count and share of each category in the legend.

    Args:
        category: (y, x) category array from categorise.
        labels: Label for each category: none, one per code, multiple, nodata.
        title: Plot title.
        png: Path to save the figure to.
    """
    # ponytail: colours repeat after 50 codes, group rare codes into "Other" if tiles get that busy
    palette = [*plt.get_cmap("tab10").colors, *plt.get_cmap("tab20b").colors, *plt.get_cmap("tab20c").colors]
    colours = ["white", *(palette[i % len(palette)] for i in range(len(labels) - 3)), "black", "lightgrey"]
    counts = np.bincount(category.ravel(), minlength=len(labels))
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(category, cmap=ListedColormap(colours), vmin=0, vmax=len(labels) - 1, interpolation="nearest")
    ax.set_title(title)
    fig.legend(
        handles=[
            Patch(facecolor=colour, edgecolor="black", label=f"{label}: {count:,} ({count / category.size:.1%})")
            for i, (colour, label, count) in enumerate(zip(colours, labels, counts))
            # Hide empty none, multiple and nodata entries, but keep empty codes so known codes are listed
            if count or 0 < i < len(labels) - 2
        ],
        loc="center left",
        bbox_to_anchor=(1, 0.5),
    )
    fig.savefig(png, dpi=150, bbox_inches="tight")
    plt.show()


def plot_distinct(distinct: np.ndarray, valid: np.ndarray, title: str, png: Path):
    """Map the distinct class count of each pixel on a continuous colour ramp, with invalid pixels in grey.

    Args:
        distinct: (y, x) array from distinct_classes.
        valid: Boolean (y, x) array of valid pixels.
        title: Plot title.
        png: Path to save the figure to.
    """
    cmap = plt.get_cmap("viridis").with_extremes(bad="lightgrey")
    fig, ax = plt.subplots(figsize=(8, 8))
    image = ax.imshow(np.ma.masked_where(~valid, distinct), cmap=cmap, vmin=1, interpolation="nearest")
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label="Distinct classes", shrink=0.8)
    fig.savefig(png, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
rows = {}
flags_by_tile = {}
flicker_by_tile = {}
implausible_by_tile = {}
distinct_by_tile = {}
stability_by_tile = {}
areas_by_tile = {}
years_by_tile = {}
lulc_by_tile = {}

for x, y in TILES:
    lulc = load_tile(x, y)
    lulc_by_tile[(x, y)] = lulc
    # Counts use the raw data so filled years are not counted
    counts = class_counts(lulc.values)
    filled = fill_missing(lulc.values)
    flickers = flicker_events(filled)
    implausibles = implausible_events(filled)
    flags = flag_pixels(filled, counts, flickers, implausibles)
    flags_by_tile[(x, y)] = flags
    distinct_by_tile[(x, y)] = distinct_classes(counts)
    stability_by_tile[f"{x:03}_{y:03}"] = class_stability(counts, flags["valid"])
    flicker_by_tile[(x, y)] = transition_categories(filled, flickers, flags["valid"])
    implausible_by_tile[(x, y)] = transition_categories(filled, implausibles, flags["valid"], IMPLAUSIBLE)
    years_by_tile[(x, y)] = lulc.time.dt.year.values.tolist()
    rows[f"{x:03}_{y:03}"] = {k: int(v.sum()) for k, v in flags.items()}

    # Reproject to equal area EPSG:6933 for class areas
    equal_area = lulc.odc.reproject(f"EPSG:{EPSG_CODE}", resolution=30, resampling="nearest", dst_nodata=NODATA)
    pixel_km2 = 30 * 30 / 1e6
    areas_by_tile[(x, y)] = pd.DataFrame(
        {classes_flipped[c]: (equal_area == c).sum(dim=["x", "y"]).values * pixel_km2 for c in CLASS_CODES},
        index=lulc.time.dt.year.values,
    )

summary = pd.DataFrame.from_dict(rows, orient="index")
summary.index.name = "tile"
class_stability_table = pd.concat(stability_by_tile, names=["tile", "class"])
summary

Pixels not temporally stable per tile, by test. A pixel can fail several tests, so the test columns do not sum to `any`.
Nodata years are skipped. Pixels with no data in any year are excluded (`valid` is the number of pixels tested).

In [ ]:
pct = (summary.drop(columns="valid").div(summary["valid"], axis=0) * 100).round(1)
pct

In [ ]:
# Map of unstable pixels per tile by test, to find hotspots
tests = summary.columns.drop(["any", "valid"])
unstable_pngs = {}
for tile, flags in flags_by_tile.items():
    # Encode the combination of failed tests as a bitmask so each combination gets its own category
    mask = sum(flags[name].astype(int) << bit for bit, name in enumerate(tests))
    category, found = categorise(np.where(mask > 0, mask, -1)[None], flags["valid"])
    unstable_pngs[tile] = REPORT_DIR / f"unstable_pixels_{tile[0]:03}_{tile[1]:03}.png"
    combos = [" + ".join(name for bit, name in enumerate(tests) if code >> bit & 1) for code in found]
    labels = ["Stable", *combos, "Multiple tests", "Nodata"]
    plot_categories(category, labels, f"Unstable pixels, tile {tile[0]:03}_{tile[1]:03}", unstable_pngs[tile])

In [ ]:
# Map of flicker class per tile, to see which classes the model confuses
flicker_pngs = {}
for tile, (category, found) in flicker_by_tile.items():
    pairs = [(classes_flipped[code // 256], classes_flipped[code % 256]) for code in found]
    labels = ["No flicker", *[f"{src} -> {mid} -> {src}" for src, mid in pairs], "Multiple", "Nodata"]
    flicker_pngs[tile] = REPORT_DIR / f"flicker_classes_{tile[0]:03}_{tile[1]:03}.png"
    plot_categories(category, labels, f"Flicker classes, tile {tile[0]:03}_{tile[1]:03}", flicker_pngs[tile])

In [ ]:
# Map of implausible transitions per tile
implausible_pngs = {}
for tile, (category, found) in implausible_by_tile.items():
    pairs = [(classes_flipped[code // 256], classes_flipped[code % 256]) for code in found]
    labels = ["None", *[f"{src} -> {dst}" for src, dst in pairs], "Multiple", "Nodata"]
    implausible_pngs[tile] = REPORT_DIR / f"implausible_transitions_{tile[0]:03}_{tile[1]:03}.png"
    plot_categories(
        category, labels, f"Implausible transitions, tile {tile[0]:03}_{tile[1]:03}", implausible_pngs[tile]
    )

Mean modal share by modal class. Lower values mean pixels of that class spend more years in other classes.

In [ ]:
class_stability_table

In [ ]:
# Map of distinct classes per pixel, to find mixed pixels and unstable hotspots
distinct_pngs = {}
for tile, distinct in distinct_by_tile.items():
    distinct_pngs[tile] = REPORT_DIR / f"distinct_classes_{tile[0]:03}_{tile[1]:03}.png"
    plot_distinct(
        distinct, flags_by_tile[tile]["valid"], f"Distinct classes, tile {tile[0]:03}_{tile[1]:03}", distinct_pngs[tile]
    )

In [ ]:
# Class area time series. Look for jumps that coincide with Landsat sensor changes or GeoMAD observation counts.
area_pngs = {}
for tile, areas in areas_by_tile.items():
    # One panel per class with its own y axis, so jumps in small classes are visible next to large ones
    areas = areas.loc[:, areas.any()]
    area_colours = [tuple(v / 255 for v in class_rgb(classes[label])) for label in areas]
    axes = areas.plot(
        subplots=True,
        layout=(-1, 3),
        figsize=(15, 2.5 * -(-len(areas.columns) // 3)),
        sharex=True,
        title=list(areas.columns),
        legend=False,
        ylabel="Area (km2)",
        color=area_colours,
    )
    fig = axes.flat[0].figure
    fig.suptitle(f"Class area, tile {tile[0]:03}_{tile[1]:03}")
    fig.tight_layout()
    area_pngs[tile] = REPORT_DIR / f"class_area_{tile[0]:03}_{tile[1]:03}.png"
    fig.savefig(area_pngs[tile], dpi=150, bbox_inches="tight")

## GeoMAD inputs by year

The GeoMAD bands, a few key indices and the MAD bands for every year, under that year's LULC, to see whether input
changes explain the LULC changes. Each row uses the same colour scale for every year.

In [ ]:
GEOMAD_VERSION = "0-2-1"  # The GeoMAD version LULC 0-1-1 was classified from
GEOMAD_PREFIX = dataset_prefix(OWNER, SENSOR, "geomad")
GEOMAD_PARQUET = f"{BUCKET_URL}/{GEOMAD_PREFIX}/{GEOMAD_VERSION}/{GEOMAD_PREFIX}.parquet"
# Inputs per figure. Each figure has the LULC as its first row, for comparison.
INPUT_FIGURES = {"spectral": ["rgb", "ndvi", "mndwi", "bsi"], "mad": ["emad", "smad", "bcmad"]}
# Landsat sensors in each year's geomedian (Landsat 7 SLC-off from mid 2003).
SENSOR_ERAS = [(2000, 2011, "L5 + L7"), (2012, 2012, "L7"), (2013, 2021, "L7 + L8"), (2022, 2025, "L8 + L9")]


def load_geomad(x: int, y: int, lulc) -> xr.Dataset:
    """Load every year's scaled GeoMAD bands and indices on the LULC grid, nodata outside the LULC area.

    Args:
        x: Tile x index.
        y: Tile y index.
        lulc: LULC (time, y, x) DataArray for the tile, from load_tile.

    Returns:
        Dataset with a time dimension, reflectance scaled to [0, 1] and NaN where LULC is nodata in every year.
    """
    ids = [f"{GEOMAD_PREFIX}_{x:03}_{y:03}_{year}" for year in YEARS]
    items = [Item.from_dict(doc) for doc in search_sync(GEOMAD_PARQUET, ids=ids)]
    if len(items) != len(ids):
        raise ValueError(f"Found {len(items)} of {len(ids)} GeoMAD items for tile {x:03}_{y:03}")
    raw = stac_load(items, bands=GEOMAD_BANDS, geobox=lulc.odc.geobox, chunks={}).load()
    data = _calculate_indices(_scale_offset_landsat(raw))
    return data.where((lulc != NODATA).any("time"))


geomad_by_tile = {tile: load_geomad(*tile, lulc_by_tile[tile]) for tile in TILES}

In [ ]:
def input_image(data: xr.Dataset, name: str, limits: tuple[float, float]) -> np.ndarray:
    """One year's image of an input: an RGB array for "rgb", otherwise the band.

    Args:
        data: One year of load_geomad output.
        name: Input name in INPUTS.
        limits: Colour scale minimum and maximum (reflectance for "rgb").

    Returns:
        (y, x, 3) RGB array in [0, 1] with white outside the data, or the (y, x) band.
    """
    if name != "rgb":
        return data[name].values
    rgb = np.stack([data[band].values for band in ["red", "green", "blue"]], axis=-1)
    return np.nan_to_num(np.clip((rgb - limits[0]) / (limits[1] - limits[0]), 0, 1), nan=1)


def input_limits(geomad: xr.Dataset, name: str) -> tuple[float, float]:
    """Colour scale for an input: its 2nd to 98th percentile over all years (and the RGB bands together)."""
    bands = ["red", "green", "blue"] if name == "rgb" else [name]
    values = np.concatenate([geomad[band].values.ravel() for band in bands])
    return tuple(np.nanpercentile(values, [2, 98]))


def save_png(fig, png: Path, dpi: int):
    """Save a figure as a 256 colour PNG, which keeps report figures small (under the 500 KB commit limit) with no
    visible loss.

    Args:
        fig: Figure to save.
        png: Path to save the figure to.
        dpi: Resolution.
    """
    buffer = io.BytesIO()
    fig.savefig(buffer, dpi=dpi, bbox_inches="tight")
    Image.open(buffer).convert("RGB").quantize(256).save(png, optimize=True)


def plot_inputs(geomad: xr.Dataset, lulc, inputs: list[str], title: str, png: Path):
    """Plot every year (columns) of the LULC (first row) and each input (other rows), each input with one colour
    scale for all years.

    Args:
        geomad: load_geomad output.
        lulc: LULC (time, y, x) DataArray on the same grid.
        inputs: Input names, e.g. from INPUT_FIGURES.
        title: Plot title.
        png: Path to save the figure to.
    """
    years = geomad.time.dt.year.values
    rows = len(inputs) + 1
    fig, axes = plt.subplots(rows, len(years), figsize=(1.6 * len(years), 1.7 * rows), squeeze=False)
    rgba = class_rgba()
    for col, layer in enumerate(lulc):
        ax = axes[0, col]
        ax.imshow(rgba[layer.values], interpolation="nearest")
        ax.set_title(str(years[col]))
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0, 0].set_ylabel("LULC")
    # Hidden colour bar so the LULC columns line up with the input rows, which have colour bars.
    fig.colorbar(plt.cm.ScalarMappable(), ax=axes[0, :].tolist(), fraction=0.01, pad=0.005).ax.set_visible(False)
    axes[0, -1].legend(
        handles=[Patch(facecolor=rgba[c] / 255, edgecolor="black", label=classes_flipped[c]) for c in CLASS_CODES],
        loc="center left",
        bbox_to_anchor=(1.05, 0.5),
        fontsize="small",
    )
    for row, name in enumerate(inputs, start=1):
        limits = input_limits(geomad, name)
        cmap = plt.get_cmap("viridis").with_extremes(bad="white")
        for col, year in enumerate(years):
            ax = axes[row, col]
            image = ax.imshow(
                input_image(geomad.isel(time=col), name, limits),
                cmap=cmap,
                vmin=limits[0],
                vmax=limits[1],
                # Antialiased, as panels are smaller than the data. Nearest speckles and makes large PNGs.
                interpolation="antialiased",
            )
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(name)
        colorbar = fig.colorbar(image, ax=axes[row, :].tolist(), fraction=0.01, pad=0.005)
        colorbar.ax.set_visible(name != "rgb")
    fig.suptitle(title)
    save_png(fig, png, 80)
    plt.show()


input_pngs = {}
for tile, geomad in geomad_by_tile.items():
    for figure, inputs in INPUT_FIGURES.items():
        input_pngs[(tile, figure)] = REPORT_DIR / f"lulc_{figure}_years_{tile[0]:03}_{tile[1]:03}.png"
        title = (
            f"LULC {LULC_VERSION} and GeoMAD {GEOMAD_VERSION} {figure} inputs by year, tile {tile[0]:03}_{tile[1]:03}"
        )
        plot_inputs(geomad, lulc_by_tile[tile], inputs, title, input_pngs[(tile, figure)])

In [ ]:
# Median of each input per year, grouped by each pixel's 2020 class. Fixing the class separates input changes on the
# same land from changes in the class mix. Shaded bands are the sensor eras.
def plot_input_medians(geomad: xr.Dataset, lulc, title: str, png: Path):
    """Plot the per-year median of each input for each 2020 class, with sensor eras shaded.

    Args:
        geomad: load_geomad output.
        lulc: LULC (time, y, x) DataArray on the same grid.
        title: Plot title.
        png: Path to save the figure to.
    """
    # Bands instead of the RGB image
    names = ["red", "nir08", "swir16", *[name for inputs in INPUT_FIGURES.values() for name in inputs if name != "rgb"]]
    lulc_2020 = lulc.sel(time=lulc.time.dt.year == 2020).squeeze("time", drop=True)
    years = geomad.time.dt.year.values
    fig, axes = plt.subplots(-(-len(names) // 3), 3, figsize=(15, 2.6 * -(-len(names) // 3)), sharex=True)
    for ax, name in zip(axes.flat, names):
        for i, (start, end, _) in enumerate(SENSOR_ERAS):
            ax.axvspan(start - 0.5, end + 0.5, alpha=0.08 * (i % 2 == 0), color="black")
        for code in CLASS_CODES:
            mask = lulc_2020 == code
            if int(mask.sum()) < 100:  # Skip classes too small for a stable median
                continue
            medians = geomad[name].where(mask).median(["y", "x"]).values
            ax.plot(years, medians, color=tuple(v / 255 for v in class_rgb(code)), label=classes_flipped[code])
        ax.set_title(name)
    for ax in axes.flat[len(names) :]:
        ax.set_visible(False)
    for ax in axes.flat[: len(names)]:
        for start, end, label in SENSOR_ERAS:
            ax.text((start + end) / 2, 0.02, label, transform=ax.get_xaxis_transform(), ha="center", fontsize="x-small")
    axes.flat[0].legend(fontsize="small")
    fig.suptitle(title)
    fig.tight_layout()
    save_png(fig, png, 150)
    plt.show()


input_median_pngs = {}
for tile, geomad in geomad_by_tile.items():
    input_median_pngs[tile] = REPORT_DIR / f"geomad_medians_{tile[0]:03}_{tile[1]:03}.png"
    plot_input_medians(
        geomad,
        lulc_by_tile[tile],
        f"GeoMAD {GEOMAD_VERSION} median by year and 2020 class, tile {tile[0]:03}_{tile[1]:03}",
        input_median_pngs[tile],
    )

In [ ]:
# Spearman correlation across years between each input's land median and the share of pixels in a class. Shows which
# inputs track the LULC changes over the years.
CORRELATION_INPUTS = [*REFLECTANCE_BANDS, "ndvi", "mndwi", "bsi", *MAD_BANDS]
CORRELATION_SHARES = {
    "built-up + other": [classes["Built-up"], classes["Other"]],
    "tree cover": [classes["Tree cover"]],
    "grassland": [classes["Grassland"]],
}


def input_correlations(geomad: xr.Dataset, lulc) -> pd.DataFrame:
    """Spearman correlation across years between each input's land median and the share of pixels in each class group.

    Args:
        geomad: load_geomad output.
        lulc: LULC (time, y, x) DataArray on the same grid.

    Returns:
        Table with a row per input and a column per class group in CORRELATION_SHARES, sorted by the strongest
        correlation with the first group.
    """
    land = (lulc != NODATA) & (lulc != classes["Water"])
    valid = (lulc != NODATA).sum(["y", "x"])
    shares = {name: (lulc.isin(codes).sum(["y", "x"]) / valid).values for name, codes in CORRELATION_SHARES.items()}
    medians = {name: geomad[name].where(land).median(["y", "x"]).values for name in CORRELATION_INPUTS}
    table = pd.DataFrame(
        {
            share: [spearmanr(medians[name], values).statistic for name in CORRELATION_INPUTS]
            for share, values in shares.items()
        },
        index=pd.Index(CORRELATION_INPUTS, name="input"),
    ).round(2)
    return table.reindex(table.iloc[:, 0].abs().sort_values(ascending=False).index)


correlations_by_tile = {tile: input_correlations(geomad, lulc_by_tile[tile]) for tile, geomad in geomad_by_tile.items()}
for tile, table in correlations_by_tile.items():
    print(f"Tile {tile[0]:03}_{tile[1]:03}")
    display(table)

## Write the report

In [ ]:
def markdown_table(df: pd.DataFrame) -> str:
    """Render a DataFrame as a markdown table.

    Args:
        df: Table to render. The index is written as columns.

    Returns:
        Markdown table string.
    """
    df = df.reset_index().rename(columns={"index": ""})
    rows = [list(map(str, df.columns)), ["-"] * len(df.columns), *df.astype(str).values.tolist()]
    return "\n".join(f"| {' | '.join(row)} |" for row in rows)


years_loaded = pd.DataFrame(
    {
        "years loaded": [len(years) for years in years_by_tile.values()],
        "years expected": len(YEARS),
        "years": [f"{min(years)}-{max(years)}" if years else "" for years in years_by_tile.values()],
    },
    index=[f"{x:03}_{y:03}" for x, y in years_by_tile],
)
settings = pd.Series(
    {
        "flip_rate": f"more than {FLIP_THRESHOLD} year-to-year changes",
        "flicker": f"A -> B -> A where B lasts at most {FLICKER_MAX_YEARS} years",
        "implausible": "any transition in the implausible list below",
        "long_reversal_or_drift": (
            f"modal class covers less than {MODAL_THRESHOLD:.0%} of years, more than one change and no flicker"
        ),
    },
    name="flagged when",
)
implausible_list = ", ".join(f"{classes_flipped[a]} -> {classes_flipped[b]}" for a, b in IMPLAUSIBLE)
unstable_figures = "\n\n".join(
    f"![Unstable pixels {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in unstable_pngs.items()
)
flicker_figures = "\n\n".join(
    f"![Flicker classes {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in flicker_pngs.items()
)
implausible_figures = "\n\n".join(
    f"![Implausible transitions {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in implausible_pngs.items()
)
distinct_figures = "\n\n".join(
    f"![Distinct classes {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in distinct_pngs.items()
)
area_figures = "\n\n".join(f"![Class area {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in area_pngs.items())
input_figures = "\n\n".join(
    f"![LULC and GeoMAD {figure} inputs by year {tile[0]:03}_{tile[1]:03}]({png.name})"
    for (tile, figure), png in input_pngs.items()
)
input_median_figures = "\n\n".join(
    f"![GeoMAD medians {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in input_median_pngs.items()
)
correlation_tables = "\n\n".join(
    f"Tile {tile[0]:03}_{tile[1]:03}\n\n{markdown_table(table)}" for tile, table in correlations_by_tile.items()
)
few_years_note = (
    f"\n**Warning:** some tiles have fewer than {len(YEARS)} years of LULC, so the flip rate and flicker tests "
    "are weak.\n"
    if (years_loaded["years loaded"] < len(YEARS)).any()
    else ""
)

report = f"""# LULC temporal stability report: version {LULC_VERSION}

- LULC: {LULC_PREFIX} version {LULC_VERSION}
- Tiles: {", ".join(years_loaded.index)}
{few_years_note}
## Years loaded

{markdown_table(years_loaded)}

## Tests

{markdown_table(settings.to_frame())}

Implausible transitions: {implausible_list}.

## Unstable pixels per tile (count)

A pixel can fail several tests, so the test columns do not sum to `any`. `valid` is the number of pixels tested.

{markdown_table(summary)}

## Unstable pixels per tile (% of valid pixels)

{markdown_table(pct)}

## Unstable pixel map

{unstable_figures}

## Flicker classes

Each A -> B -> A event is found at the first year the pixel returns to A.

{flicker_figures}

## Implausible transitions

{implausible_figures}

## Class stability

Mean share of years each pixel is in its modal class, grouped by modal class. Lower values mean pixels of that class
spend more years in other classes, from real change or misclassification.

{markdown_table(class_stability_table)}

## Distinct classes per pixel

{distinct_figures}

## Class area time series

Year-to-year jumps that coincide with Landsat sensor changes or low GeoMAD observation counts are artefacts, not
land change.

{area_figures}

## LULC and GeoMAD inputs by year

LULC with the GeoMAD {GEOMAD_VERSION} bands, key indices and MAD bands it was classified from. Each input row uses one
colour scale for all years.

{input_figures}

Median of each input per year for the pixels of each 2020 class. Steps that coincide with sensor changes (shaded
eras) point to input artefacts rather than land change.

{input_median_figures}

## Inputs that track the LULC changes

Spearman correlation across years between each input's median over land (water excluded) and the share of pixels in
each class group. Inputs with strong correlations follow the year-to-year LULC changes.

{correlation_tables}

## Caveats

- Nodata years are skipped, so year-to-year tests compare each valid year with the previous valid year.
Pixels with no valid years are not tested.
- Tiles are cropped and masked to {", ".join(COUNTRY)} buffered by {COUNTRY_BUFFER_M} m.
- Class areas are from LULC reprojected to equal area EPSG:6933 at 30 m with nearest neighbour resampling.
"""
report_path = REPORT_DIR / "temporal_stability.md"
report_path.write_text(report)
print(f"Wrote {report_path}")